# 62 — Outcome Maturation & Label Lock

Construye labels únicamente cuando cada contrato tenga los **180 días completos de follow-up**. Los labels se congelan con manifest y hash antes de calcular métricas.

No deben imputarse negativos a contratos inmaduros.


In [ ]:
from argos.common.paths import ProjectPaths
from argos.config.prospective import load_prospective_spec
from argos.config.production import load_production_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
M7_SPEC = load_prospective_spec(paths.configs / "prospective.toml")
M6_SPEC = load_production_spec(paths.configs / "production.toml")
(paths.reports / "prospective").mkdir(parents=True, exist_ok=True)
(paths.root / "artifacts" / "prospective").mkdir(parents=True, exist_ok=True)

m6_gate = paths.docs / "decisions" / "M6_GATE.md"
assert m6_gate.exists() and "M6 ✅ CLOSED" in m6_gate.read_text(encoding="utf-8"), "M6 must be CLOSED before M7."
print("M7 schema:", M7_SPEC.schema_version)
print("Protocol freeze:", M7_SPEC.core.protocol_freeze_date)
print("Prospective start:", M7_SPEC.core.prospective_start)


In [ ]:
from datetime import timedelta

first_possible_maturity = M7_SPEC.core.prospective_start + timedelta(days=M7_SPEC.core.target_horizon_days)
print("Earliest contractual maturity date:", first_possible_maturity)
print("Outcome reporting lag buffer (days):", M7_SPEC.cohorts.outcome_reporting_lag_days)
print("Required matured cohorts:", M7_SPEC.cohorts.min_matured_cohorts)
print("Required matured rows:", M7_SPEC.cohorts.min_matured_rows)
print("Required positives:", M7_SPEC.cohorts.min_matured_positives)


In [ ]:
# Execution contract for the future run:
# 1. Read only batches registered by notebook 61.
# 2. Use a versioned future Silver addition-events snapshot.
# 3. Compute an outcome watermark = max(event_at) - reporting-lag buffer.
# 4. Mature only rows where t0 + 180 days <= outcome watermark.
# 5. Build y_addition_180d with the frozen window (t0, t0 + 180d].
# 6. Persist immutable label files + manifests before any performance calculation.
# 7. Never reinterpret immature contracts as negatives.
#
# The repository intentionally contains no synthetic future outcomes.
print("62 ⏳ WAITING FOR MATURED PROSPECTIVE OUTCOMES")
